# EDA - AI4I 2020 Predictive Maintenance Dataset

Este notebook realiza uma análise exploratória inicial do dataset **AI4I 2020 Predictive Maintenance** e justifica as features usadas no pipeline de engenharia de atributos do projeto.

Objetivo do projeto:

- prever a probabilidade de falha de uma máquina/processo industrial;
- usar `Machine failure` como alvo principal de classificação binária;
- usar `TWF`, `HDF`, `PWF`, `OSF` e `RNF` apenas para diagnóstico, análise e segmentação, evitando vazamento de informação no modelo principal;
- preparar o caminho para monitoramento de drift e governança com MLflow/Evidently.

Arquivo analisado:

```text
../data/raw/ai4i2020.csv
```

## 1. Imports e configuração

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

DATA_PATH = Path("../data/raw/ai4i2020.csv")
DATA_PATH.exists(), DATA_PATH

## 2. Carregamento do dataset

O arquivo pode conter BOM UTF-8 no cabeçalho, por isso usamos `encoding="utf-8-sig"`.

In [ ]:
df_raw = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
df_raw.head()

In [ ]:
df_raw.shape

In [ ]:
df_raw.info()

## 3. Contrato esperado de colunas

As colunas esperadas para o dataset AI4I são listadas abaixo. O pipeline de ingestão trata `HDF`, `PWF`, `OSF` e `RNF` como colunas opcionais, porque alguns arquivos podem conter apenas `TWF` como modo de falha adicional. Quando ausentes, essas colunas são preenchidas com `0` no carregamento.

In [ ]:
required_columns = [
    "UDI",
    "Product ID",
    "Type",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
    "Machine failure",
    "TWF",
]

optional_failure_mode_columns = ["HDF", "PWF", "OSF", "RNF"]

missing_required = [column for column in required_columns if column not in df_raw.columns]
missing_optional = [column for column in optional_failure_mode_columns if column not in df_raw.columns]

missing_required, missing_optional

In [ ]:
df = df_raw.copy()

for column in optional_failure_mode_columns:
    if column not in df.columns:
        df[column] = 0

df.columns.tolist()

## 4. Qualidade dos dados

Nesta etapa validamos valores nulos, duplicidade de `UDI`, tipos de produto e consistência básica dos rótulos binários.

In [ ]:
quality_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": df.isna().mean() * 100,
    "dtype": df.dtypes.astype(str),
})

quality_summary

In [ ]:
checks = {
    "rows": len(df),
    "duplicated_udi": int(df["UDI"].duplicated().sum()),
    "product_types": sorted(df["Type"].dropna().unique().tolist()),
    "machine_failure_values": sorted(df["Machine failure"].dropna().unique().tolist()),
}

checks

## 5. Distribuição do alvo

O problema é desbalanceado: falhas são eventos raros em relação ao total de observações. Isso impacta a escolha de métricas e modelos.

Métricas recomendadas:

- `Recall` da classe de falha;
- `Precision` da classe de falha;
- `F1`;
- `ROC AUC`;
- `PR AUC`;
- matriz de confusão.

Em manutenção preditiva, falsos negativos podem ser caros, pois significam deixar de identificar uma falha real.

In [ ]:
target_counts = df["Machine failure"].value_counts().sort_index()
target_pct = df["Machine failure"].value_counts(normalize=True).sort_index() * 100

pd.DataFrame({"count": target_counts, "pct": target_pct})

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
target_counts.plot(kind="bar", ax=ax)
ax.set_title("Distribuição do alvo: Machine failure")
ax.set_xlabel("Machine failure")
ax.set_ylabel("Quantidade")
ax.set_xticklabels(["Sem falha", "Falha"], rotation=0)
plt.show()

## 6. Modos de falha

`TWF`, `HDF`, `PWF`, `OSF` e `RNF` são modos de falha. Eles explicam o tipo de falha, mas **não devem entrar como features do classificador principal**, porque representam informação de resultado/evento.

Uso adequado dessas colunas:

- análise exploratória;
- avaliação por fatias;
- explicações no Model Card;
- modelo diagnóstico separado, caso o objetivo seja classificar o tipo de falha após identificar uma falha.

In [ ]:
failure_modes = ["TWF", "HDF", "PWF", "OSF", "RNF"]
failure_mode_counts = df[failure_modes].sum().sort_values(ascending=False)
failure_mode_counts

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
failure_mode_counts.plot(kind="bar", ax=ax)
ax.set_title("Quantidade por modo de falha")
ax.set_xlabel("Modo de falha")
ax.set_ylabel("Quantidade")
plt.show()

## 7. Distribuição por tipo de produto

`Type` representa a variante de qualidade do produto (`L`, `M`, `H`). Essa informação é disponível antes do evento de falha e pode ser usada como feature categórica.

In [ ]:
type_summary = pd.crosstab(df["Type"], df["Machine failure"], margins=True)
type_summary

In [ ]:
type_failure_rate = df.groupby("Type")["Machine failure"].mean().sort_values(ascending=False)
type_failure_rate

## 8. Estatísticas das variáveis numéricas

As variáveis numéricas de processo/sensor são as principais entradas do modelo. Elas representam temperatura, velocidade, torque e desgaste da ferramenta.

In [ ]:
numeric_columns = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]

df[numeric_columns].describe().T

In [ ]:
fig, axes = plt.subplots(len(numeric_columns), 1, figsize=(10, 14))

for ax, column in zip(axes, numeric_columns):
    ax.hist(df[column], bins=40)
    ax.set_title(f"Distribuição - {column}")
    ax.set_xlabel(column)
    ax.set_ylabel("Frequência")

plt.tight_layout()
plt.show()

## 9. Comparação das variáveis por falha

A comparação entre registros com e sem falha ajuda a justificar features derivadas. Por exemplo, torque, velocidade, desgaste e diferença de temperatura estão diretamente relacionados aos modos de falha descritos no dataset.

In [ ]:
df.groupby("Machine failure")[numeric_columns].agg(["mean", "median", "std"]).T

In [ ]:
fig, axes = plt.subplots(1, len(numeric_columns), figsize=(18, 4))

for ax, column in zip(axes, numeric_columns):
    df.boxplot(column=column, by="Machine failure", ax=ax)
    ax.set_title(column)
    ax.set_xlabel("Machine failure")

plt.suptitle("Variáveis numéricas por classe do alvo")
plt.tight_layout()
plt.show()

## 10. Engenharia de features

As features abaixo são inspiradas nas regras físicas descritas no dataset e no artigo original.

Justificativas:

- `temperature_delta_k`: diferença entre temperatura de processo e temperatura do ar. A falha por dissipação de calor ocorre quando essa diferença é baixa e a rotação é baixa.
- `rotational_speed_rad_s`: converte RPM para rad/s, necessário para cálculo aproximado de potência.
- `power_w`: aproxima potência como torque vezes velocidade angular. A falha de potência ocorre em faixas anormais de potência.
- `tool_wear_by_torque`: aproxima esforço acumulado da ferramenta.
- `overstrain_margin`: compara esforço com limites específicos por tipo de produto.
- flags de limiar: traduzem regras físicas conhecidas em features interpretáveis.
- one-hot de `Type`: representa diferenças de variante/qualidade do produto.

In [ ]:
def normalize_columns(raw_df: pd.DataFrame) -> pd.DataFrame:
    column_map = {
        "UDI": "udi",
        "Product ID": "product_id",
        "Type": "product_type",
        "Air temperature [K]": "air_temperature_k",
        "Process temperature [K]": "process_temperature_k",
        "Rotational speed [rpm]": "rotational_speed_rpm",
        "Torque [Nm]": "torque_nm",
        "Tool wear [min]": "tool_wear_min",
        "Machine failure": "machine_failure",
        "TWF": "twf",
        "HDF": "hdf",
        "PWF": "pwf",
        "OSF": "osf",
        "RNF": "rnf",
    }
    normalized = raw_df.copy()
    for optional_column in ["HDF", "PWF", "OSF", "RNF"]:
        if optional_column not in normalized.columns:
            normalized[optional_column] = 0
    return normalized.rename(columns=column_map)


def build_ai4i_features(df: pd.DataFrame) -> pd.DataFrame:
    features = df.copy()
    thresholds = {"L": 11000.0, "M": 12000.0, "H": 13000.0}

    features["temperature_delta_k"] = features["process_temperature_k"] - features["air_temperature_k"]
    features["rotational_speed_rad_s"] = features["rotational_speed_rpm"] * 2.0 * np.pi / 60.0
    features["power_w"] = features["torque_nm"] * features["rotational_speed_rad_s"]
    features["torque_speed_interaction"] = features["torque_nm"] * features["rotational_speed_rpm"]
    features["tool_wear_by_torque"] = features["tool_wear_min"] * features["torque_nm"]
    features["temperature_delta_low_flag"] = (
        (features["temperature_delta_k"] < 8.6)
        & (features["rotational_speed_rpm"] < 1380)
    ).astype(int)
    features["power_low_flag"] = (features["power_w"] < 3500).astype(int)
    features["power_high_flag"] = (features["power_w"] > 9000).astype(int)
    features["overstrain_threshold"] = features["product_type"].map(thresholds)
    features["overstrain_margin"] = features["tool_wear_by_torque"] - features["overstrain_threshold"]

    type_dummies = pd.get_dummies(features["product_type"], prefix="type", dtype=int)
    for column in ["type_H", "type_L", "type_M"]:
        if column not in type_dummies:
            type_dummies[column] = 0

    return pd.concat([features, type_dummies[["type_H", "type_L", "type_M"]]], axis=1)


df_normalized = normalize_columns(df)
df_features = build_ai4i_features(df_normalized)
df_features.head()

In [ ]:
engineered_columns = [
    "temperature_delta_k",
    "rotational_speed_rad_s",
    "power_w",
    "torque_speed_interaction",
    "tool_wear_by_torque",
    "temperature_delta_low_flag",
    "power_low_flag",
    "power_high_flag",
    "overstrain_margin",
    "type_H",
    "type_L",
    "type_M",
]

df_features[engineered_columns].describe().T

## 11. Features derivadas vs alvo

Abaixo comparamos as features derivadas entre registros com e sem falha. Isso ajuda a verificar se as features capturam sinais relacionados ao risco operacional.

In [ ]:
df_features.groupby("machine_failure")[engineered_columns].agg(["mean", "median", "std"]).T

In [ ]:
selected_engineered_columns = [
    "temperature_delta_k",
    "power_w",
    "tool_wear_by_torque",
    "overstrain_margin",
]

fig, axes = plt.subplots(1, len(selected_engineered_columns), figsize=(18, 4))

for ax, column in zip(axes, selected_engineered_columns):
    df_features.boxplot(column=column, by="machine_failure", ax=ax)
    ax.set_title(column)
    ax.set_xlabel("machine_failure")

plt.suptitle("Features derivadas por classe do alvo")
plt.tight_layout()
plt.show()

## 12. Correlação entre variáveis

A matriz abaixo não deve ser interpretada como causalidade, mas ajuda a identificar relações lineares entre variáveis e possíveis redundâncias.

Importante: os modos de falha (`twf`, `hdf`, `pwf`, `osf`, `rnf`) não devem entrar no conjunto de features do modelo primário, pois são rótulos diagnósticos do evento.

In [ ]:
correlation_columns = [
    "air_temperature_k",
    "process_temperature_k",
    "rotational_speed_rpm",
    "torque_nm",
    "tool_wear_min",
    "temperature_delta_k",
    "power_w",
    "tool_wear_by_torque",
    "overstrain_margin",
    "machine_failure",
]

corr = df_features[correlation_columns].corr(numeric_only=True)
corr

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(correlation_columns)))
ax.set_yticks(range(len(correlation_columns)))
ax.set_xticklabels(correlation_columns, rotation=90)
ax.set_yticklabels(correlation_columns)
fig.colorbar(im, ax=ax)
ax.set_title("Matriz de correlação")
plt.tight_layout()
plt.show()

## 13. Conjunto de features recomendado para o modelo inicial

Para o classificador primário de `Machine failure`, o conjunto inicial deve conter apenas atributos conhecidos antes da falha:

- `air_temperature_k`
- `process_temperature_k`
- `temperature_delta_k`
- `rotational_speed_rpm`
- `rotational_speed_rad_s`
- `torque_nm`
- `tool_wear_min`
- `power_w`
- `torque_speed_interaction`
- `tool_wear_by_torque`
- `temperature_delta_low_flag`
- `power_low_flag`
- `power_high_flag`
- `overstrain_margin`
- `type_H`, `type_L`, `type_M`

Não usar como entrada:

- `machine_failure`
- `twf`, `hdf`, `pwf`, `osf`, `rnf`
- `UDI`
- `Product ID`

`UDI` e `Product ID` são identificadores e não devem carregar sinal operacional generalizável para o modelo.

In [ ]:
model_feature_columns = [
    "air_temperature_k",
    "process_temperature_k",
    "temperature_delta_k",
    "rotational_speed_rpm",
    "rotational_speed_rad_s",
    "torque_nm",
    "tool_wear_min",
    "power_w",
    "torque_speed_interaction",
    "tool_wear_by_torque",
    "temperature_delta_low_flag",
    "power_low_flag",
    "power_high_flag",
    "overstrain_margin",
    "type_H",
    "type_L",
    "type_M",
]

X = df_features[model_feature_columns]
y = df_features["machine_failure"]

X.shape, y.shape, y.mean()

## 14. Implicações para drift

Como o dataset original é estático, o projeto deve tratar a ingestão como batch versionado. Para demonstrar drift, podemos comparar:

- uma partição de referência vs uma partição atual/holdout;
- o dataset original vs novos arquivos CSV colocados em `data/incoming/`;
- distribuições de features brutas e derivadas;
- distribuição de probabilidades previstas pelo modelo;
- métricas de performance quando rótulos estiverem disponíveis.

Features especialmente relevantes para drift:

- `air_temperature_k`
- `process_temperature_k`
- `temperature_delta_k`
- `rotational_speed_rpm`
- `torque_nm`
- `tool_wear_min`
- `power_w`
- `overstrain_margin`
- `type_*`

## 15. Conclusões

Principais decisões justificadas pela EDA:

1. O problema principal é uma classificação binária desbalanceada, então recall, precision, F1, ROC AUC e PR AUC são métricas mais adequadas do que acurácia isolada.
2. Os modos de falha devem ser usados para diagnóstico e análise, mas não como features do classificador primário para evitar vazamento.
3. Features físicas como potência, diferença de temperatura e margem de sobrecarga são justificadas pela própria definição dos modos de falha.
4. `Type` deve ser codificado como variável categórica, pois altera limites e comportamento de desgaste.
5. O pipeline de ingestão deve preservar dados brutos versionados e gerar uma tabela de features reproduzível para treino, serving e drift.

Próximo passo recomendado:

- treinar uma baseline de Regressão Logística com `class_weight='balanced'`;
- comparar com Random Forest ou Gradient Boosting;
- registrar runs, métricas e artefatos no MLflow;
- criar relatório de drift com Evidently usando uma partição de referência.